## 3. Pipeline de extração de vocabulário

Regras de elegibilidade para um *token* entrar na contagem (unigrama) ou compor um bigrama:

- classe gramatical em `{NOUN, PROPN, VERB, ADJ}` (substantivos, nomes próprios, verbos, adjetivos);
- não ser stopword padrão do spaCy, nem constar nas camadas 2 e 3 acima;
- ser alfabético (aceitando hífen interno, para preservar compostos como *open-source*);
- comprimento mínimo de 2 caracteres.

Um bigrama só é formado quando os dois tokens adjacentes (na mesma frase) atendem a todas essas regras simultaneamente.

Para entrar no ranking final, exigimos um mínimo de **3 ocorrências** para unigramas e **2 ocorrências** para bigramas (bigramas são naturalmente mais raros) — um piso mínimo de confiabilidade estatística, especialmente relevante para `ai_plus.json`, o documento mais curto do corpus (~2.900 palavras).

# Análise Lexical dos Planos Nacionais/Regionais de Inteligência Artificial

Este notebook analisa o **vocabulário** de 6 documentos oficiais de estratégia/plano de ação para Inteligência Artificial, publicados por diferentes países e blocos:

| Arquivo | Documento | País/Bloco | Publicação |
|---|---|---|---|
| `ai_continent_action_plan.json` | AI Continent Action Plan | Europa (Comissão Europeia) | 2025-04 |
| `apply_ai_strategy.json` | Apply AI Strategy | Europa (Comissão Europeia) | 2025-10 |
| `ai_plus.json` | Opinions on Deepening "Artificial Intelligence+" | China (Conselho de Estado) | 2025-08 |
| `new_generation_ai_development_plan.json` | New Generation AI Development Plan | China (Conselho de Estado) | 2017-07 |
| `americas_ai_action_plan.json` | America's AI Action Plan | Estados Unidos (Casa Branca) | 2025-07 |
| `pbia.json` | AI for the Good of All (PBIA) | Brasil (MCTI/CGEE) | 2025 |

## Objetivo

1. Levantar o **Top 50** de vocábulos/termos (unigramas e bigramas) mais utilizados por documento, com a frequência **normalizada por 1.000 tokens** (para permitir comparação justa entre documentos de tamanhos muito diferentes — de ~2.900 a ~17.700 palavras).
2. Levantar **15 termos adicionais mais específicos/distintivos** de cada documento — termos usados com ênfase real por poucos documentos (não pela maioria do corpus).
3. Tratar explicitamente **bigramas** (ex.: *artificial intelligence*, *open-source*, *national security*) e **termos ambíguos** (*intelligence* vs. *intelligent*, *safe* vs. *safety*, *risk* vs. *risks*), evitando tanto a fragmentação artificial de contagens quanto a fusão indevida de palavras com significados diferentes.
4. Construir **stopwords efetivas**, específicas para este corpus e, quando necessário, específicas por documento (nome do próprio país/bloco, ministérios e instituições próprias, iniciativas próprias, e artefatos de formatação/template do PDF de origem).

In [ ]:
import json
import re
import glob
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display, Markdown
from wordcloud import WordCloud

import spacy
from spacy.util import compile_infix_regex

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

# Garante que o modelo em inglês do spaCy está disponível (usado para
# lematização com reconhecimento de classe gramatical — ver seção de
# metodologia abaixo).
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

nlp.add_pipe("sentencizer")

# Por padrão, o tokenizer do spaCy separa palavras compostas por hífen
# em 3 tokens (ex.: "open-source" -> "open", "-", "source"). Como o
# usuário pediu atenção a termos como "Open-Source", removemos a regra
# de infixo que quebra em hífen, preservando compostos como um único
# token: open-source, human-centric, high-risk, state-of-the-art...
infixes = [pat for pat in nlp.Defaults.infixes if "-" not in pat]
nlp.tokenizer.infix_finditer = compile_infix_regex(infixes).finditer

DOCS_DIR = "/home/lantri_leonardoferreira/codigo/Fundamentos/notebooks/ Novo/"
FILES = sorted(f.split("/")[-1] for f in glob.glob(DOCS_DIR + "*.json"))
FILES

## 1. Carregamento do corpus

Cada JSON já passou por uma etapa prévia de extração cuidadosa (documentada no próprio campo `elementos_descartados`), que removeu capas, sumários, notas de rodapé, numeração de página etc. Aqui carregamos apenas o texto corrido (`texto_completo`) e os metadados de cada documento.

In [ ]:
raw_docs = {}
for fname in FILES:
    with open(DOCS_DIR + fname, encoding="utf-8") as fh:
        raw_docs[fname] = json.load(fh)

resumo = pd.DataFrame([
    {
        "arquivo": f,
        "país/bloco": d["pais_ou_bloco"],
        "título": d["titulo"][:60] + ("…" if len(d["titulo"]) > 60 else ""),
        "publicação": d["data_publicacao"],
        "caracteres": len(d["texto_completo"]),
    }
    for f, d in raw_docs.items()
])
display(resumo)

## 2. Metodologia

### 2.1 Lematização com reconhecimento de classe gramatical (evitando fundir termos ambíguos)

Em vez de um *stemmer* (que corta sufixos de forma cega), usamos o **lematizador do spaCy**, que leva em conta a classe gramatical (POS) de cada palavra. Isso resolve exatamente o problema de termos ambíguos citado no pedido:

- `risks` → `risk` (plural → singular: **correto fundir**, é a mesma palavra/sentido)
- `safe` (adjetivo) e `safety` (substantivo) → lemas **diferentes** (`safe`, `safety`): **correto manter separados**, pois são conceitos distintos (algo pode ser "seguro" sem que o documento fale de "segurança" como área/governança)
- `intelligence` (substantivo) e `intelligent` (adjetivo) → lemas **diferentes**: idem — "inteligência" (a capacidade/o campo) não é o mesmo que "inteligente" (o qualificativo aplicado a sistemas)

Ou seja: um *stemmer* tradicional erraria nos dois últimos casos (fundiria `safe`/`safety` e `intelligence`/`intelligent` num único radical); a lematização por POS resolve isso automaticamente. Uma seção dedicada (§6) compara explicitamente essas famílias de termos ambíguos entre os 6 documentos.

### 2.2 Processamento em minúsculas (correção de um viés do lematizador)

Documentos de política pública são cheios de títulos e rótulos em *Title Case* ("Recommended Policy Actions", "Key Commission Actions:"). O lematizador do spaCy trata palavras capitalizadas em início de frase/título como possíveis nomes próprios e, nesses casos, **não** as lematiza (ex.: "Actions" no cabeçalho ficaria como "Actions", diferente de "action" no corpo do texto — inflando artificialmente a contagem de termos únicos). Por isso o texto é processado em minúsculas antes da lematização, o que elimina essa fragmentação e é seguro aqui porque toda a nossa contagem já é *case-insensitive*.

### 2.3 Unigramas *e* bigramas

Muitos dos conceitos mais importantes destes planos são expressões de duas palavras (*artificial intelligence*, *open-source*, *national security*, *machine learning*). Por isso construímos, para cada documento, tanto a lista de unigramas quanto a de bigramas — mas só formamos um bigrama quando **as duas palavras são adjacentes no texto original e ambas são palavras de conteúdo** (não stopwords), evitando bigramas artificiais formados a partir de palavras que não estavam realmente uma ao lado da outra.

### 2.4 Frequência normalizada por 1.000 tokens

Os documentos variam de ~2.900 a ~17.700 palavras — quase 6x de diferença. Comparar contagens brutas favoreceria sempre os documentos mais longos. Por isso todas as frequências são reportadas como

    taxa = (nº de ocorrências do termo / nº total de palavras do documento) × 1.000

um padrão comum em linguística de corpus (frequência "por mil palavras"), que é o que efetivamente ordena os rankings de Top 50 e Top 15 abaixo.

### 2.5 Stopwords efetivas (três camadas)

Usamos três camadas de exclusão, todas listadas explicitamente no código (nada "caixa-preta"):

1. **Stopwords linguísticas padrão** do spaCy (artigos, preposições, pronomes, conjunções) + filtro por classe gramatical: só entram na análise substantivos, nomes próprios, verbos e adjetivos. Isso já elimina, de graça, praticamente todos os conectivos/marcadores discursivos citados no pedido (*meanwhile*, *although*, *however*, *therefore*, *furthermore*...), pois são advérbios/conjunções.
2. **Stopwords genéricas de conteúdo** (`GENERAL_EXTRA_STOPWORDS`): verbos/adjetivos que aparecem de forma praticamente universal em qualquer documento de política pública e não diferenciam nada (*ensure*, *include*, *provide*, *support*, *new*, *key*...).
3. **Stopwords específicas por documento** (`CUSTOM_STOPWORDS`): nome do próprio país/bloco e seus gentílicos (ex.: *Brazil/Brazilian* só é removido do PBIA; permanece visível nos demais 5 documentos, onde sua menção *é* informativa), ministérios/instituições/órgãos próprios (MCTI, CGEE, State Council, European Commission, DOC, DOE, NIST...) e autorreferências ao próprio plano/iniciativa (ex.: "PBIA", "Apply AI Strategy").

Além disso, tratamos **artefatos estruturais de formatação** (`STRUCTURAL_CLEAN`): alguns documentos usam rótulos-modelo repetidos dezenas de vezes (ex.: o PBIA descreve cada uma das suas ~80 medidas com os campos fixos "Action N:", "Challenge:", "Expected impact(s):"; o plano americano introduz cada recomendação com "Led by [agência],"; o "AI Continent Action Plan" repete o cabeçalho "Key Commission Actions:"). Sem tratamento, essas palavras de *formatação* dominariam o ranking de vocabulário só por causa da estrutura do documento, não do seu conteúdo — por isso os rótulos são removidos do texto antes da análise (o conteúdo que vem depois deles é preservado).

In [ ]:
# ------------------------------------------------------------------
# Limpeza estrutural específica por documento: remove rótulos/templates
# repetidos (formatação do PDF de origem), preservando o conteúdo que
# vem em seguida. Ver §2.5.
# ------------------------------------------------------------------
STRUCTURAL_CLEAN = {
    "pbia.json": [
        # cada uma das ~80 medidas do plano é descrita com campos fixos
        (r"\bImpact Action \d+:\s*", " "),
        (r"\bAction \d+:\s*", " "),
        (r"\bChallenge:\s*", " "),
        (r"\bExpected impacts?:\s*", " "),
    ],
    "americas_ai_action_plan.json": [
        # cabeçalho de seção repetido 30x + prefixo de atribuição de
        # cada recomendação a uma agência ("Led by DOC, ...")
        (r"\bRecommended Policy Actions\b", " "),
        (r"\bLed by\b", " "),
    ],
    "ai_continent_action_plan.json": [
        (r"Key Commission(?:\s*/\s*EuroHPC)?\s+[Aa]ctions:?", " "),
    ],
}

for fname, patterns in STRUCTURAL_CLEAN.items():
    text = raw_docs[fname]["texto_completo"]
    for pat, repl in patterns:
        text = re.sub(pat, repl, text)
    raw_docs[fname]["texto_limpo"] = text
for fname in FILES:
    raw_docs[fname].setdefault("texto_limpo", raw_docs[fname]["texto_completo"])

In [ ]:
# ------------------------------------------------------------------
# Camada 2: stopwords genéricas de conteúdo (verbos/adjetivos "vazios",
# comuns a qualquer documento de política pública, que não diferenciam
# nada e só inflam o ranking). Ver §2.5.
# ------------------------------------------------------------------
GENERAL_EXTRA_STOPWORDS = {
    "include", "ensure", "provide", "support", "need", "new", "key",
    "major", "relevant", "set", "work", "well", "particular", "certain",
    "various", "significant", "given", "existing", "exist",
}

# ------------------------------------------------------------------
# Camada 3: stopwords específicas por documento — nome do próprio
# país/bloco e gentílico, ministérios/instituições/órgãos próprios,
# autorreferências ao próprio plano. Cada item abaixo foi conferido
# manualmente no texto-fonte antes de ser incluído.
# ------------------------------------------------------------------
CUSTOM_STOPWORDS = {
    "ai_continent_action_plan.json": {
        # UE/Comissão Europeia (bloco e instituição próprios)
        "european", "europe", "eu", "union", "commission", "member",
        "communication",  # "this Communication" = autorreferência ao tipo de ato
        "state",           # "Member State(s)" = autorreferência institucional
        "apply",           # "(the) Apply AI Strategy" = iniciativa-irmã da própria UE
    },
    "apply_ai_strategy.json": {
        "european", "europe", "eu", "union", "commission", "member",
        "communication", "state",
        "apply",  # "Apply AI Strategy" = autorreferência ao próprio plano
    },
    "ai_plus.json": {
        "china", "chinese",         # país/gentílico próprio
        "council", "state",         # "State Council" = órgão emissor próprio
    },
    "new_generation_ai_development_plan.json": {
        "china", "chinese",
        "council",             # "State Council"
        "party", "committee",  # "CCP/CPC Central Committee", "Party Congress"
    },
    "americas_ai_action_plan.json": {
        "america", "american",   # país/gentílico próprio
        "united", "states",      # "United States"
        "trump", "administration",  # "(the) Trump Administration" = governo em exercício
        # agências/órgãos federais próprios (siglas)
        "doc", "doe", "nsf", "caisi", "ostp", "dol", "dod", "nsc", "dos",
        "cte", "dhs", "nairr", "ftc", "ntia", "bls", "bea", "nstc", "nist",
        "gsa", "nepa", "odni", "cbrne",
    },
    "pbia.json": {
        "brazil", "brazilian",  # país/gentílico próprio
        # instituições/siglas próprias do plano
        "mcti", "cgee", "cct", "abc", "pbia", "finep", "mgi",
    },
}

In [ ]:
ALLOWED_POS = {"NOUN", "PROPN", "VERB", "ADJ"}
WORD_RE = re.compile(r"^[a-z][a-z\-]*[a-z]$|^[a-z]$")
MIN_LEN = 2
MIN_COUNT_UNI = 3
MIN_COUNT_BI = 2

# Correções manuais de lematização: o lematizador do spaCy erra o
# sufixo de palavras compostas fora do seu vocabulário (ex.: o plural
# "gigafactories" vira "gigafactorie" em vez de "gigafactory").
# Conferido manualmente contra o texto-fonte de cada documento.
LEMMA_FIX = {
    "gigafactorie": "gigafactory",
    "giga-factorie": "gigafactory",
    "specie": "species",
}

# Ajustes só de exibição: siglas (para não aparecerem como "Ai", "Eu")
# e o caso "datum", lema gramaticalmente correto de "data" que soaria
# estranho numa tabela de vocabulário.
DISPLAY_OVERRIDES = {
    "ai": "AI", "eu": "EU", "us": "US", "gdp": "GDP", "r&d": "R&D",
    "nist": "NIST", "hpc": "HPC", "gpu": "GPU", "gpus": "GPU", "ict": "ICT",
    "sme": "SME", "smes": "SMEs", "sus": "SUS", "llm": "LLM", "llms": "LLMs",
    "5g": "5G", "eurohpc": "EuroHPC", "iot": "IoT", "api": "API",
    "datum": "Data",
}

def display_term(term: str) -> str:
    return " ".join(DISPLAY_OVERRIDES.get(w, w.capitalize()) for w in term.split(" "))

def clean_lemma(tok) -> str:
    lemma = tok.lemma_.lower()
    return LEMMA_FIX.get(lemma, lemma)

def is_content(tok, stop_extra) -> bool:
    if tok.pos_ not in ALLOWED_POS or tok.is_stop:
        return False
    lemma = clean_lemma(tok)
    if not WORD_RE.match(lemma) or len(lemma) < MIN_LEN:
        return False
    if lemma in GENERAL_EXTRA_STOPWORDS or lemma in stop_extra:
        return False
    return True

def analyze(fname: str) -> pd.DataFrame:
    text = raw_docs[fname]["texto_limpo"].lower()
    stop_extra = CUSTOM_STOPWORDS.get(fname, set())
    doc = nlp(text)

    total_words = sum(1 for t in doc if WORD_RE.match(t.text) and len(t.text) >= MIN_LEN)

    uni, bi = Counter(), Counter()
    for sent in doc.sents:
        toks = list(sent)
        flags = [is_content(t, stop_extra) for t in toks]
        lemmas = [clean_lemma(t) for t in toks]
        for i, ok in enumerate(flags):
            if ok:
                uni[lemmas[i]] += 1
        for i in range(len(toks) - 1):
            if flags[i] and flags[i + 1]:
                bi[f"{lemmas[i]} {lemmas[i+1]}"] += 1

    rows = [(t, "unigrama", n) for t, n in uni.items() if n >= MIN_COUNT_UNI]
    rows += [(t, "bigrama", n) for t, n in bi.items() if n >= MIN_COUNT_BI]

    df = pd.DataFrame(rows, columns=["termo", "tipo", "ocorrencias"])
    df["por_1000_tokens"] = df["ocorrencias"] / total_words * 1000
    df["termo_exibicao"] = df["termo"].apply(display_term)
    df = df.sort_values("por_1000_tokens", ascending=False).reset_index(drop=True)
    df.attrs["total_words"] = total_words
    return df

results = {fname: analyze(fname) for fname in FILES}

pd.DataFrame([
    {"arquivo": f, "país/bloco": raw_docs[f]["pais_ou_bloco"],
     "total de palavras": results[f].attrs["total_words"],
     "termos elegíveis (uni+bi)": len(results[f])}
    for f in FILES
])

## 4. Top 50 — vocabulário principal de cada documento

Ranking único combinando unigramas e bigramas elegíveis, ordenado pela taxa de ocorrência por 1.000 tokens (coluna `tipo` indica se o termo é uma palavra ou uma expressão de duas palavras).

In [ ]:
TOP_N_VOCAB = 50

def show_table(df, n, cols_rename):
    out = df.head(n).copy()
    out.insert(0, "rank", range(1, len(out) + 1))
    out = out[["rank", "termo_exibicao", "tipo", "ocorrencias", "por_1000_tokens"]]
    out["por_1000_tokens"] = out["por_1000_tokens"].round(2)
    out.columns = cols_rename
    return out.set_index("rank")

COLS = ["rank", "Termo", "Tipo", "Ocorrências", "Por 1.000 tokens"]

for fname in FILES:
    d = raw_docs[fname]
    display(Markdown(f"### {d['titulo']}  \n*{d['pais_ou_bloco']} — {fname}*"))
    display(show_table(results[fname], TOP_N_VOCAB, COLS))

## 5. Top 15 — termos mais distintivos/específicos de cada documento

Aqui o objetivo muda: não é "o que mais aparece" e sim **"o que diferencia este documento dos demais"**. A regra segue exatamente o critério pedido — importa a *ênfase relativa*, não a aparição isolada:

1. Um termo é considerado usado **"com ênfase"** num documento se estiver entre os `TOP_K_PRESENCE = 200` termos desse documento (por taxa/1.000 tokens) — ou seja, faz parte do vocabulário realmente relevante daquele texto, não uma menção de passagem.
2. Contamos em **quantos dos 6 documentos** cada termo atinge esse patamar de ênfase (`presence_count`).
3. Um termo só é candidato a "distintivo" se `presence_count <= 3` — ou seja, aparece com ênfase em **no máximo metade** dos documentos. Se aparece com ênfase em 4 ou mais (maioria/quase todos), é vocabulário comum da área de política de IA, não uma especificidade do documento, e é descartado.
4. Dentro dos candidatos de cada documento, ordenamos pela taxa de ocorrência **naquele documento** e pegamos os 15 primeiros.

Isso reproduz fielmente o exemplo dado: um termo como *"open-source"*, usado com ênfase por apenas 1–3 documentos, aparece na lista; um termo usado com ênfase por 4+ documentos (ex.: *"national"*, *"development"*), por mais frequente que seja, não aparece aqui — ele já está retratado no Top 50 de cada um, mas não é o que os diferencia entre si.

In [ ]:
TOP_K_PRESENCE = 200
TOP_N_DISTINCTIVE = 15

doc_topsets = {
    fname: dict(zip(results[fname].head(TOP_K_PRESENCE)["termo"],
                     results[fname].head(TOP_K_PRESENCE)["por_1000_tokens"]))
    for fname in FILES
}

presence_count = Counter()
for fname in FILES:
    for termo in doc_topsets[fname]:
        presence_count[termo] += 1

distinctive = {}
for fname in FILES:
    df = results[fname]
    mask = df["termo"].apply(lambda t: t in doc_topsets[fname] and presence_count[t] <= 3)
    dsub = df[mask].head(TOP_N_DISTINCTIVE).copy()
    dsub["presente_em_n_docs"] = dsub["termo"].map(presence_count)
    distinctive[fname] = dsub.reset_index(drop=True)

# diagnóstico: quantos termos do "vocabulário relevante" (top-200 de
# pelo menos 1 doc) são compartilhados por quantos documentos
diag = pd.Series(Counter(presence_count.values())).sort_index()
diag.index.name = "presente com ênfase em N documentos"
diag.name = "nº de termos"
diag

In [ ]:
COLS_DIST = ["rank", "Termo", "Tipo", "Ocorrências", "Por 1.000 tokens", "Presente c/ ênfase em N docs"]

for fname in FILES:
    d = raw_docs[fname]
    out = distinctive[fname].copy()
    out.insert(0, "rank", range(1, len(out) + 1))
    out = out[["rank", "termo_exibicao", "tipo", "ocorrencias", "por_1000_tokens", "presente_em_n_docs"]]
    out["por_1000_tokens"] = out["por_1000_tokens"].round(2)
    out.columns = COLS_DIST
    display(Markdown(f"### {d['titulo']}  \n*{d['pais_ou_bloco']} — {fname}*"))
    display(out.set_index("rank"))

## 6. Visualizações

### 6.1 Termos mais distintivos por documento

Um gráfico por documento com seus 15 termos mais específicos (§5), em barras horizontais ordenadas pela taxa de ocorrência. Cada documento recebe uma cor fixa, usada de forma consistente em todos os gráficos deste notebook.

In [ ]:
# Paleta categórica (ordem fixa, uma cor por documento — nunca reciclada)
DOC_COLOR = {
    "ai_continent_action_plan.json":          "#2a78d6",  # azul
    "apply_ai_strategy.json":                 "#eb6834",  # laranja
    "ai_plus.json":                           "#1baf7a",  # água
    "new_generation_ai_development_plan.json":"#eda100",  # amarelo
    "americas_ai_action_plan.json":           "#4a3aa7",  # violeta
    "pbia.json":                              "#e34948",  # vermelho
}
INK = "#0b0b0b"
MUTED = "#898781"
GRID = "#e1e0d9"

# Títulos curtos (só ASCII) para uso em gráficos matplotlib: os títulos
# completos de alguns documentos incluem o nome original em chinês
# entre parênteses, que a fonte padrão do matplotlib não renderiza.
SHORT_TITLE = {
    "ai_continent_action_plan.json": "AI Continent Action Plan",
    "apply_ai_strategy.json": "Apply AI Strategy",
    "ai_plus.json": 'Opinions on "Artificial Intelligence+"',
    "new_generation_ai_development_plan.json": "New Generation AI Development Plan (2017)",
    "americas_ai_action_plan.json": "America's AI Action Plan",
    "pbia.json": "PBIA — AI for the Good of All",
}

# Rótulo curto e único por documento (usado em eixos de gráfico) — como
# há 2 documentos da Europa e 2 da China, "país/bloco" sozinho não
# distingue qual dos dois é cada barra.
SHORT_LABEL = {
    "ai_continent_action_plan.json": "EU · Continent",
    "apply_ai_strategy.json": "EU · Apply AI",
    "ai_plus.json": "China · AI+",
    "new_generation_ai_development_plan.json": "China · 2017",
    "americas_ai_action_plan.json": "EUA",
    "pbia.json": "Brasil",
}

fig, axes = plt.subplots(3, 2, figsize=(13, 16))
for ax, fname in zip(axes.flat, FILES):
    d = raw_docs[fname]
    dsub = distinctive[fname].iloc[::-1]  # menor->maior para barh
    color = DOC_COLOR[fname]
    bars = ax.barh(dsub["termo_exibicao"], dsub["por_1000_tokens"], color=color, height=0.68)
    for bar, val in zip(bars, dsub["por_1000_tokens"]):
        ax.text(bar.get_width() + max(dsub["por_1000_tokens"]) * 0.02, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", ha="left", fontsize=8.5, color=INK)
    ax.set_title(f"{SHORT_TITLE[fname]}\n({d['pais_ou_bloco']})", fontsize=10.5, color=INK, loc="left")
    ax.set_xlabel("ocorrências por 1.000 tokens", fontsize=8.5, color=MUTED)
    ax.tick_params(axis="y", labelsize=8.7, colors=INK)
    ax.tick_params(axis="x", labelsize=8, colors=MUTED)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.set_xlim(0, max(dsub["por_1000_tokens"]) * 1.18)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle("Top 15 termos mais distintivos por documento", fontsize=14, color=INK, y=1.0)
fig.tight_layout()
plt.show()

### 6.2 Nuvens de palavras (Top 50 de cada documento)

Visão complementar e mais intuitiva do vocabulário *principal* (não do distintivo) de cada documento — o tamanho de cada termo é proporcional à sua taxa de ocorrência por 1.000 tokens.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

fig, axes = plt.subplots(3, 2, figsize=(13, 15))
for ax, fname in zip(axes.flat, FILES):
    d = raw_docs[fname]
    freqs = dict(zip(results[fname].head(TOP_N_VOCAB)["termo_exibicao"],
                      results[fname].head(TOP_N_VOCAB)["por_1000_tokens"]))
    cmap = LinearSegmentedColormap.from_list("doc_cmap", ["#c3c2b7", DOC_COLOR[fname]])
    def color_func(word=None, font_size=None, position=None, orientation=None,
                    random_state=None, cmap=cmap, freqs=freqs, **kwargs):
        maxv = max(freqs.values())
        t = 0.35 + 0.65 * (freqs.get(word, 0) / maxv)
        r, g, b, _ = cmap(t)
        return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"

    wc = WordCloud(width=900, height=650, background_color="#fcfcfb",
                    prefer_horizontal=0.92, color_func=color_func,
                    max_words=TOP_N_VOCAB, relative_scaling=0.6,
                    margin=4).generate_from_frequencies(freqs)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{SHORT_TITLE[fname]}\n({d['pais_ou_bloco']})", fontsize=10.5, color=INK)

fig.suptitle("Top 50 — nuvem de vocabulário por documento", fontsize=14, color=INK, y=1.0)
fig.tight_layout()
plt.show()

## 7. Foco: famílias de termos ambíguos

Comparação direta, entre os 6 documentos, das famílias de termos citadas no pedido — palavras com a mesma raiz mas classes gramaticais/sentidos diferentes, que a lematização por POS manteve **separadas** (ver §2.1). Isso permite ver, por exemplo, se um documento fala mais de "risco" (substantivo, governança) do que de algo ser "arriscado" (adjetivo), ou se prioriza "inteligência" (o campo/capacidade) sobre sistemas "inteligentes" (o qualificativo) — uma escolha de enquadramento que os números brutos do Top 50 não deixam tão evidente lado a lado.

In [ ]:
TERM_FAMILIES = {
    "intelligence / intelligent": ["intelligence", "intelligent"],
    "safe / safety": ["safe", "safety"],
    "risk / risky": ["risk", "risky"],
    "secure / security": ["secure", "security"],
    "regulate / regulation / regulatory": ["regulate", "regulation", "regulatory"],
    "govern / governance": ["govern", "governance"],
    "innovate / innovation / innovative": ["innovate", "innovation", "innovative"],
    "open / open-source": ["open", "open-source"],
    "smart / intelligentize": ["smart", "intelligentize"],
}

rows = []
for fam, terms in TERM_FAMILIES.items():
    for fname in FILES:
        df = results[fname]
        for term in terms:
            hit = df.loc[df["termo"] == term, "por_1000_tokens"]
            rows.append({
                "família": fam, "termo": display_term(term),
                "documento": raw_docs[fname]["pais_ou_bloco"] + " — " + raw_docs[fname]["titulo"][:28],
                "arquivo": fname,
                "por_1000_tokens": float(hit.iloc[0]) if len(hit) else 0.0,
            })
fam_df = pd.DataFrame(rows)

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, (fam, terms) in zip(axes.flat, TERM_FAMILIES.items()):
    sub = fam_df[fam_df["família"] == fam]
    piv = sub.pivot(index="arquivo", columns="termo", values="por_1000_tokens").reindex(FILES)
    piv.index = [SHORT_LABEL[f] for f in FILES]
    piv.plot(kind="bar", ax=ax, color=[DOC_COLOR["ai_continent_action_plan.json"],
                                        DOC_COLOR["apply_ai_strategy.json"],
                                        DOC_COLOR["ai_plus.json"]][:piv.shape[1]],
              width=0.75, legend=True)
    ax.set_title(fam, fontsize=10, color=INK)
    ax.set_xlabel("")
    ax.set_ylabel("por 1.000 tokens", fontsize=8, color=MUTED)
    ax.tick_params(axis="x", labelrotation=45, labelsize=7.5, colors=INK)
    ax.tick_params(axis="y", labelsize=7.5, colors=MUTED)
    ax.legend(fontsize=7.5, frameon=False)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

fig.suptitle("Famílias de termos ambíguos — taxa por 1.000 tokens, por documento", fontsize=13, color=INK, y=1.02)
fig.tight_layout()
plt.show()

## 8. Exportação (opcional)

Salva o Top 50 e o Top 15 distintivo de cada documento em CSV, na própria pasta `Novo/`, para consulta fora do notebook.

In [ ]:
OUT_DIR = DOCS_DIR

for fname in FILES:
    base = fname.replace(".json", "")
    show_table(results[fname], TOP_N_VOCAB, COLS).to_csv(f"{OUT_DIR}{base}_top50_vocabulario.csv", encoding="utf-8-sig")
    out = distinctive[fname].copy()
    out.insert(0, "rank", range(1, len(out) + 1))
    out = out[["rank", "termo_exibicao", "tipo", "ocorrencias", "por_1000_tokens", "presente_em_n_docs"]]
    out.columns = COLS_DIST
    out.set_index("rank").to_csv(f"{OUT_DIR}{base}_top15_distintivos.csv", encoding="utf-8-sig")

print("CSVs exportados para:", OUT_DIR)

## 9. Limitações metodológicas

- **Traduções**: `ai_plus.json` e `new_generation_ai_development_plan.json` são traduções para o inglês (CSET e New America/DigiChina, respectivamente) de originais em chinês. O vocabulário capturado aqui reflete também escolhas dos tradutores (ex.: "intelligentize", "smart" como tradução de 智能化/智慧), não só o texto-fonte original — uma camada de viés que não existe nos demais 4 documentos, escritos originalmente em inglês/português.
- **Limiares ajustáveis**: os parâmetros `TOP_K_PRESENCE` (§5, "o que conta como uso com ênfase") e os mínimos de ocorrência (`MIN_COUNT_UNI`/`MIN_COUNT_BI`) foram calibrados observando a distribuição real do corpus, mas são escolhas metodológicas, não valores universais — o notebook foi construído para que sejam fáceis de alterar e re-executar.
- **Stopwords específicas por documento** (`CUSTOM_STOPWORDS`) foram levantadas manualmente a partir de leitura do texto-fonte; documentos futuros adicionados ao corpus exigem a mesma checagem manual antes de entrar na análise.
- **Bigramas** exigem adjacência direta no texto original; expressões de 3+ palavras (ex.: "high-risk AI systems") não são capturadas como unidade única — apenas seus pares adjacentes.